In [1]:
import polars as pl 
from pathlib import Path 
import logging
import json
import time
import re
import requests
import os
import subprocess
from collections import defaultdict
from tqdm import tqdm

import gemmi



logger = logging.getLogger(__name__)

STEP_X3_OUTPUT_PATH = Path("/cluster/project/beltrao/kdammer/master_thesis/data/Pipeline/t11_RM_TM_updated_CF_pipeline/cf_pdb_structure_similarity/aggregate_cf_for_pdb_eval_with_combfold_results_paths.parquet") 
REFERENCE_PDB_DIR = Path("/cluster/project/beltrao/kdammer/master_thesis/data/reference_pdb")

In [2]:
df_stepX3 = pl.read_parquet(STEP_X3_OUTPUT_PATH)

In [3]:
print(df_stepX3.columns)

['complex_ac', 'identifiers', 'pred_1', 'pred_2', 'pred_3', 'correct_pred_count', 'correct_pred_rank', 'CF_1_n_assemblies', 'CF_1_confidence', 'CF_1_reason', 'CF_2_n_assemblies', 'CF_2_confidence', 'CF_2_reason', 'CF_3_n_assemblies', 'CF_3_confidence', 'CF_3_reason', 'CF_true_n_assemblies', 'CF_true_confidence', 'CF_true_reason', 'combfold_job_ids', 'n_pairs_used', 'pairs_used', 'n_proteins', 'pdb_id', 'match_class', 'pdb_stoichiometry', 'CP_stochiometry', 'reference_pdb_model', 'found_match_reference_pdb_model', 'Combfold_result_path', 'n_combfold_outputs']


In [4]:
assert df_stepX3["found_match_reference_pdb_model"].all(), "Not all rows have a matching reference PDB model. Please check the data. This should have been filtered in the previous step X3."
df_relevant_cols = df_stepX3[["complex_ac", "identifiers", "n_proteins", "pdb_id", "reference_pdb_model", "CP_stochiometry", "Combfold_result_path", "n_pairs_used", "pairs_used", "correct_pred_rank", "n_combfold_outputs"]]

In [5]:
def _get_reference_pdb_path(pdb_id: str, reference_pdb_model: str, reference_pdb_dir: Path = REFERENCE_PDB_DIR) -> Path:
    """
    Given a PDB ID and a reference PDB model, return the path to the corresponding reference PDB file.
    """
    pdb_id = pdb_id.lower()
    if reference_pdb_model == "au":
        file_path_without_suffix =  (reference_pdb_dir / pdb_id / pdb_id)
    else:
        file_path_without_suffix = (reference_pdb_dir / pdb_id / f"{pdb_id}-assembly{reference_pdb_model}")

    if file_path_without_suffix.with_suffix(".pdb").exists():
        return file_path_without_suffix.with_suffix(".pdb")
    elif file_path_without_suffix.with_suffix(".cif").exists():
        return file_path_without_suffix.with_suffix(".cif")
    raise FileNotFoundError(f"Neither .pdb nor .cif file found for {pdb_id} at {file_path_without_suffix}.")

# resolve reference pdb path per row 
df_relevant_cols = df_relevant_cols.with_columns(
    pl.struct(["pdb_id", "reference_pdb_model"])
    .map_elements(
        lambda row: str(_get_reference_pdb_path(row["pdb_id"], row["reference_pdb_model"])),
        return_dtype=pl.Utf8,
    )
    .alias("reference_pdb_path")
)

In [6]:
def _get_combfold_output_paths(combfold_result_path: str, n_combfold_outputs: int) -> list[Path]:
    """
    Given a Combfold_result_path, list the assembled CombFold output files under
    `{combfold_result_path}/assembled_results/`, sorted for a stable/reproducible order.

    Raises if the number of files found does not match `n_combfold_outputs` -- we don't want
    to silently proceed with a mismatched count.
    """
    combfold_result_path = Path(combfold_result_path)
    if n_combfold_outputs == 0:
        return []

    assembled_results_dir = combfold_result_path / "assembled_results"

    if not assembled_results_dir.is_dir():
        raise FileNotFoundError(f"assembled_results dir does not exist: {assembled_results_dir}")

    assert all(p.suffix in (".txt", ".pdb") for p in assembled_results_dir.iterdir()), f"Unexpected file extension found in {assembled_results_dir}. Expected only .txt or .pdb files."
    output_paths = sorted(assembled_results_dir.glob("*.pdb"))

    if len(output_paths) != n_combfold_outputs:
        raise ValueError(
            f"Expected {n_combfold_outputs} CombFold outputs in {assembled_results_dir}, "
            f"found {len(output_paths)}."
        )

    return output_paths

# %%
# resolve the list of CombFold output paths per row, then explode into one row per output
df_relevant_cols = df_relevant_cols.with_columns(
    pl.struct(["Combfold_result_path", "n_combfold_outputs"])
    .map_elements(
        lambda row: [
            str(p)
            for p in _get_combfold_output_paths(row["Combfold_result_path"], row["n_combfold_outputs"])
        ],
        return_dtype=pl.List(pl.Utf8),
    )
    .alias("combfold_output_path")
)

df_long = df_relevant_cols.explode("combfold_output_path")

# sanity checks: row count should scale by n_combfold_outputs (treating 0 -> 1 row from explode),
# and only rows with n_combfold_outputs == 0 should have a null combfold_output_path
expected_n_rows = (
    df_relevant_cols["n_combfold_outputs"].clip(lower_bound=1).sum()
)
assert df_long.height == expected_n_rows, (
    f"Expected {expected_n_rows} rows after exploding, got {df_long.height}."
)

expected_null_outputs = (df_relevant_cols["n_combfold_outputs"] == 0).sum()
actual_null_outputs = df_long["combfold_output_path"].null_count()
assert actual_null_outputs == expected_null_outputs, (
    f"Expected {expected_null_outputs} null combfold_output_path rows (n_combfold_outputs == 0), "
    f"got {actual_null_outputs}."
)

assert df_long["reference_pdb_path"].null_count() == 0, "Found null reference_pdb_path."

logger.info(f"Built long df with {df_long.height} rows from {df_relevant_cols.height} complexes.")


In [8]:
import re
from Bio.PDB import MMCIFParser, PDBParser

sifts_full = pl.read_csv(
    "/cluster/project/beltrao/kdammer/master_thesis/data/pdb/pdb_chain_uniprot.csv",
    skip_rows=1,
    columns=["PDB", "CHAIN", "SP_PRIMARY"],
).rename(str.lower)

#for efficiency only keep reelvabt odb ids for that run
RELEVANT_PDB_IDS = set(df_long["pdb_id"].str.to_lowercase().unique().to_list())
sifts_rel = sifts_full.filter(pl.col("pdb").is_in(RELEVANT_PDB_IDS))

# creates one row per (pdb_id, uniprot) pair, with a list of candidate chain IDs.
uniprot_chains_by_pdb = (
    sifts_rel.select(["pdb", "chain", "sp_primary"]).unique()
    .group_by(["pdb", "sp_primary"])
    .agg(pl.col("chain").unique().alias("chains"))
)

SIFTS_CHAIN_LOOKUP: dict[str, dict[str, list[str]]] = {}
for pdb_id, uniprot, chains in uniprot_chains_by_pdb.iter_rows():
    SIFTS_CHAIN_LOOKUP.setdefault(pdb_id, {})[uniprot] = chains


#returns chain letters/IDs that actually exist in this specific file 
# (as pdb_chain_uniprot.csv is bulit on sifts from AU, whhich can contain more chains than are in the biological assembly)
def _get_structure_chain_ids(struct_path: str) -> set[str]:
    parser = MMCIFParser(QUIET=True) if struct_path.endswith(".cif") else PDBParser(QUIET=True)
    structure = parser.get_structure("x", struct_path)
    return {chain.id for chain in structure[0]}


def _get_pdb_chain_map(pdb_id: str, assembly_chain_ids: set[str]) -> dict[str, list[str]]:
    """uniprot -> chain_ids present in this assembly (may be >1 for homomers)."""
    candidates = SIFTS_CHAIN_LOOKUP[pdb_id]
    return {unp: sorted(set(chains) & assembly_chain_ids) for unp, chains in candidates.items()}


def _build_chain_mapping(
    reference_pdb_path: str, combfold_output_path: str | None, complex_ac: str | None = None
) -> dict[str, tuple[list[str], list[str]]]:
    if combfold_output_path is None:
        logger.warning(f"{complex_ac or '?'}: no combfold_output_path (n_combfold_outputs == 0) -- empty mapping.")
        return {}

    pdb_id = Path(reference_pdb_path).parent.name
    assembly_chain_ids = _get_structure_chain_ids(reference_pdb_path)
    pdb_chains = _get_pdb_chain_map(pdb_id, assembly_chain_ids)

    chain_list = Path(combfold_output_path).parent.parent / "_unified_representation" / "assembly_output" / "chain.list"
    cf_chains: dict[str, list[str]] = {}
    for line in chain_list.read_text().split():
        m = re.match(r"^(\w+)_(\w+)\.pdb$", line)
        if m is None:
            logger.warning(f"{complex_ac or '?'}: unparseable chain.list line {line!r} in {chain_list} -- empty mapping.")
            return {}
        unp, chain = m.groups()
        cf_chains.setdefault(unp, []).append(chain)

    if set(cf_chains) != set(pdb_chains):
        logger.warning(
            f"{complex_ac or '?'}: uniprot mismatch cf vs pdb: {set(cf_chains) ^ set(pdb_chains)} -- empty mapping."
        )
        return {}

    for unp in cf_chains:
        if len(cf_chains[unp]) != len(pdb_chains[unp]):
            logger.warning(
                f"{complex_ac or '?'}: {unp} copy-number mismatch: "
                f"CF has {cf_chains[unp]}, PDB assembly has {pdb_chains[unp]} -- empty mapping."
            )
            return {}
        
    # NOTE: for homomers (len > 1), which CF chain corresponds to which PDB chain
    # is NOT resolved here -- returns both chain lists, correspondence left to the
    # RMSD step (permutation search over copies).

    return {unp: (cf_chains[unp], pdb_chains[unp]) for unp in cf_chains}


df_long = df_long.with_columns(
    pl.struct(["reference_pdb_path", "combfold_output_path", "complex_ac"])
    .map_elements(
        lambda r: _build_chain_mapping(r["reference_pdb_path"], r["combfold_output_path"], r["complex_ac"]),
        return_dtype=pl.Object,
    )
    .alias("chain_mapping")
)

CPX-940: no combfold_output_path (n_combfold_outputs == 0) -- empty mapping.


In [9]:
uniprot_chains_by_pdb.head(7)

pdb,sp_primary,chains
str,str,list[str]
"""2p22""","""Q02767""","[""B""]"
"""2qsf""","""P32628""","[""X""]"
"""5dfz""","""P22219""","[""B""]"
"""4dgw""","""Q07350""","[""C""]"
"""2qsf""","""P14736""","[""A""]"
"""5dfz""","""P22543""","[""C""]"
"""2qlv""","""P06782""","[""D"", ""A""]"


In [10]:
df_long.row(0, named=True)

{'complex_ac': 'CPX-940',
 'identifiers': 'P25604(1)|P42939(1)|Q02767(1)|Q99176(1)',
 'n_proteins': 4,
 'pdb_id': '2P22',
 'reference_pdb_model': '1',
 'CP_stochiometry': '{"P25604": 1, "P42939": 1, "Q02767": 1, "Q99176": 1}',
 'Combfold_result_path': '/cluster/project/beltrao/kdammer/master_thesis/data/Pipeline/t11_RM_TM_updated_CF_pipeline/CombFold/P25604x1_P42939x1_Q02767x1_Q99176x1_pool_output',
 'n_pairs_used': 6,
 'pairs_used': "['P25604_P25604', 'P25604_P42939', 'P25604_Q02767', 'P42939_Q02767', 'Q02767_Q02767', 'Q02767_Q99176']",
 'correct_pred_rank': '1',
 'n_combfold_outputs': 0,
 'reference_pdb_path': '/cluster/project/beltrao/kdammer/master_thesis/data/reference_pdb/2p22/2p22-assembly1.cif',
 'combfold_output_path': None,
 'chain_mapping': {}}

In [11]:
USALIGN_BIN = "/cluster/project/beltrao/kdammer/master_thesis/tools/usalign/USalign/USalign"
def _run_usalign_on_single_row(
        df_row: pl.DataFrame,
        mm: int = 1, 
        ter: int = 1
    ) -> dict:
    assert df_row.height == 1, "Expected a single row DataFrame."
    _df = df_row.row(0, named=True)

    from itertools import chain
    cf_chains =  list(chain.from_iterable([v[0] for v in _df["chain_mapping"].values()]))
    ref_chains =  list(chain.from_iterable([v[1] for v in _df["chain_mapping"].values()]))

    assert len(cf_chains) == len(ref_chains), "Mismatch in number of chains between CF and reference."
    if len(cf_chains) == 0:
        logger.warning(f"{_df['combfold_output_path']}: no chains to align -- skipping.")
        return


    # Build path where usalign outputs are stored, within the Combfold output dir for that complex
    combfold_output_dir = Path(_df["combfold_output_path"]).parent.parent
    us_align_output_basedir = combfold_output_dir / "usalign_outputs"
    os.makedirs(us_align_output_basedir, exist_ok=True) # this one ca nexist, bc shared by different pred outputs
    us_align_output_dir = us_align_output_basedir / f"usalign_outputs_pred{_df['combfold_output_path'].split('_')[-1][:-4]}"
    os.makedirs(us_align_output_dir, exist_ok=False) # this one shouldnt exist yet
    
    results = {}

    # Run for complex
    complex_pred = us_align_output_dir / "complex"
    os.mkdir(complex_pred)
    cmd = [USALIGN_BIN, _df['combfold_output_path'], _df['reference_pdb_path'],
           "-mm", str(mm), "-ter", str(ter), "-mol", "prot",
           "-chain1", ",".join(cf_chains),
           "-chain2", ",".join(ref_chains),
           "-o", str(complex_pred / "usalign"),]
    logger.info(f"Running USalign with command: {' '.join(cmd)}")
    usalign = subprocess.run(cmd, capture_output=True, text=True, check=True)
    results['complex'] = usalign.stdout
    with open(complex_pred / "usalign_stdout.txt", "w") as f:
        f.write(usalign.stdout)

    # Run separately for each chain pair
    for prot_id, (cf_chains, ref_chains) in _df['chain_mapping'].items():
        cf_chains = list(chain.from_iterable(cf_chains))
        ref_chains = list(chain.from_iterable(ref_chains))
        assert len(cf_chains) == len(ref_chains), f"Mismatch in number of chains for {prot_id} between CF and reference."
        for cf_chain, ref_chain in zip(cf_chains, ref_chains):
            chain_pred = us_align_output_dir / f"chain_{cf_chain}_{ref_chain}_{prot_id}"
            os.mkdir(chain_pred)
            cmd = [USALIGN_BIN, _df['combfold_output_path'], _df['reference_pdb_path'],
                   "-mm", str(mm), "-ter", str(ter), "-mol", "prot",
                   "-chain1", cf_chain,
                   "-chain2", ref_chain,
                   "-o", str(chain_pred / "usalign"),]
            logger.info(f"Running USalign for chain pair {cf_chain}-{ref_chain} with command: {' '.join(cmd)}")
            usalign = subprocess.run(cmd, capture_output=True, text=True, check=True)
            results[f'chain_{cf_chain}_{ref_chain}_{prot_id}'] = usalign.stdout
            with open(chain_pred / "usalign_stdout.txt", "w") as f:
                f.write(usalign.stdout)

    return results


In [12]:
from concurrent.futures import ThreadPoolExecutor

# row_dfs = [df_long[i] for i in range(df_long.height)]

with ThreadPoolExecutor(max_workers=8) as ex:
    results = list(ex.map(_run_usalign_on_single_row, row_dfs))

NameError: name 'row_dfs' is not defined

In [14]:
def parse_usalign_output(filepath: str) -> dict:
    """
    Parse a US-align stdout file and extract key metrics.

    Returns a dict with:
        - len_structure1 (int)
        - len_structure2 (int)
        - aligned_length (int)
        - rmsd (float)
        - seq_id (float)
        - tm_score (float)   # normalized by length of Structure_2
    """
    text = Path(filepath).read_text()

    patterns = {
        "len_structure1": r"Length of Structure_1:\s*(\d+)\s*residues",
        "len_structure2": r"Length of Structure_2:\s*(\d+)\s*residues",
        "aligned_length": r"Aligned length=\s*(\d+)",
        "seq_id": r"Seq_ID=n_identical/n_aligned=\s*([\d.]+)",
        "rmsd": r"RMSD=\s*([\d.]+)",
        # Only match the TM-score line explicitly normalized by Structure_2
        "tm_score": r"TM-score=\s*([\d.]+)\s*\(normalized by length of Structure_2",
    }

    result = {}
    for key, pattern in patterns.items():
        match = re.search(pattern, text)
        if match is None:
            raise ValueError(f"Could not find '{key}' in {filepath}")
        value = match.group(1)
        if key in ("len_structure1", "len_structure2", "aligned_length"):
            result[key] = int(value)
        else:
            result[key] = float(value)

    return result

In [15]:
# This is a list of paths to the cpx level usalign stdout files, which we will parse to extract the metrics.
df1 = df_long.with_columns(
    pl.col("combfold_output_path")
    .str.replace(
        r"assembled_results/output_clustered_(\d+)\.pdb$",
        "usalign_outputs/usalign_outputs_pred${1}/complex/usalign_stdout.txt",
    )
    .alias("usalign_stdout_path")
)


METRIC_DTYPES = {
    "len_structure1": pl.Int64,
    "len_structure2": pl.Int64,
    "aligned_length": pl.Int64,
    "rmsd": pl.Float64,
    "seq_id": pl.Float64,
    "tm_score": pl.Float64,
}
result_schema = pl.Struct(METRIC_DTYPES)

new_cols = ["len_structure1", "len_structure2", "aligned_length", "rmsd", "seq_id", "tm_score"]

df1 = df1.with_columns(
    pl.col("usalign_stdout_path")
    .map_elements(parse_usalign_output, return_dtype=result_schema)
    .alias("cpx")
).unnest("cpx")

df1 = df1.rename({c: f"usalign_cpx_{c}" for c in new_cols})

In [16]:
per_chain_schema = pl.Struct(
    {"usalign_per_chain_ID": pl.List(pl.Utf8)}
    | {f"usalign_per_chain_{k}": pl.List(v) for k, v in METRIC_DTYPES.items()}
)


def parse_usalign_per_chain(pred_dir: str) -> dict:
    pred_path = Path(pred_dir)
    ids: list[str] = []
    values: dict[str, list] = {k: [] for k in METRIC_DTYPES}

    if not pred_path.is_dir():
        logger.warning(f"usalign per-chain dir missing: {pred_dir}")
        return {"usalign_per_chain_ID": ids} | {
            f"usalign_per_chain_{k}": v for k, v in values.items()
        }

    chain_dirs = sorted(
        d for d in pred_path.iterdir() if d.is_dir() and d.name != "complex"
    )

    for d in chain_dirs:
        parts = d.name.split("_")
        assert len(parts) >= 4 and parts[0] == "chain", f"unexpected chain dir name: {d}"
        chain_a, chain_b, unp = parts[1], parts[2], "_".join(parts[3:])
        assert len(chain_a) == 1 and len(chain_b) == 1, f"unexpected chain letters in: {d}"
        ids.append(f"{unp}_{chain_a}_{chain_b}")

        stdout_path = d / "usalign_stdout.txt"
        try:
            parsed = parse_usalign_output(str(stdout_path))
        except Exception as e:
            logger.warning(f"failed to parse usalign output at {stdout_path}: {e}")
            parsed = {k: None for k in METRIC_DTYPES}

        for k in METRIC_DTYPES:
            values[k].append(parsed.get(k))

    return {"usalign_per_chain_ID": ids} | {
        f"usalign_per_chain_{k}": v for k, v in values.items()
    }


df1 = df1.with_columns(
    pl.col("combfold_output_path")
    .str.replace(
        r"assembled_results/output_clustered_(\d+)\.pdb$",
        "usalign_outputs/usalign_outputs_pred${1}",
    )
    .alias("usalign_pred_dir")
)

df1 = df1.with_columns(
    pl.col("usalign_pred_dir")
    .map_elements(parse_usalign_per_chain, return_dtype=per_chain_schema)
    .alias("per_chain")
).unnest("per_chain")

In [17]:
# check whether chainmap does anythign 
USALIGN_BIN = "/cluster/project/beltrao/kdammer/master_thesis/tools/usalign/USalign/USalign"

with open("chainmap.txt", "w") as f:
 f.write("A\tB\nB\tC\nC\tD\nD\tA\n")

cmd = [
 USALIGN_BIN,
 "/cluster/project/beltrao/kdammer/master_thesis/data/Pipeline/t11_RM_TM_updated_CF_pipeline/CombFold/P22219x1_P22543x1_Q02948x1_Q05919x1_pool_output/assembled_results/output_clustered_0.pdb",
 "/cluster/project/beltrao/kdammer/master_thesis/data/reference_pdb/5dfz/5dfz-assembly1.cif",
 "-mm", "1", "-ter", "0", "-mol", "prot",
 "-chain1", "A,B,C,D",
 "-chain2", "B,C,D,A",
 "-chainmap", "chainmap.txt",
 "-o", "/cluster/project/beltrao/kdammer/master_thesis/tmp",
]
# usalign = subprocess.run(cmd, capture_output=True, text=True, check=True)
# print(usalign.stdout)

Which protein chains are close to each other

In [18]:
def get_close_by_chain_pairs(path, cutoff=8.0, min_contacts=5):
    st = gemmi.read_structure(path)
    st.setup_entities()
    model = st[0]

    ns = gemmi.NeighborSearch(model, st.cell, cutoff).populate()
    cs = gemmi.ContactSearch(cutoff)
    cs.ignore = gemmi.ContactSearch.Ignore.SameChain

    contacts = cs.find_contacts(ns)

    contact_count = defaultdict(int)
    for c in contacts:
        if c.partner1.atom.name != "CA" or c.partner2.atom.name != "CA":
            continue
        pair = tuple(sorted((c.partner1.chain.name, c.partner2.chain.name)))
        contact_count[pair] += 1

    interfaces = {pair: n for pair, n in contact_count.items() if n >= min_contacts}
    return interfaces


PDB_FILE = "/cluster/project/beltrao/kdammer/master_thesis/data/Pipeline/7_benchmark_part_one/CombFold/O13539x1_P17629x1_P33441x1_P53552x1_P53851x1_pool_output/assembled_results/output_clustered_0.pdb"
get_close_by_chain_pairs(PDB_FILE)

{('A', 'C'): 46,
 ('A', 'D'): 501,
 ('A', 'B'): 548,
 ('B', 'D'): 325,
 ('B', 'C'): 191,
 ('B', 'E'): 74,
 ('C', 'D'): 417,
 ('C', 'E'): 144,
 ('D', 'E'): 105}

In [19]:
def _add_close_by_chain_pairs_columns(df_row: pl.DataFrame, cutoff: float = 8.0, min_contacts: int = 5) -> pl.DataFrame:
    assert df_row.height == 1, "Expected a single row DataFrame."
    row = df_row.row(0, named=True)

    def invert_chain_mapping_dicts(d):
        cf_to_unp = {}
        ref_to_unp = {}
        for unp_id, (cf_chains, ref_chains) in d.items():
            for chain in cf_chains:
                cf_to_unp[chain] = unp_id
            for chain in ref_chains:
                ref_to_unp[chain] = unp_id
        return cf_to_unp, ref_to_unp

    def map_chain_pairs_to_unp(pair_dict, chain_to_unp):
        return {
            f"{chain_to_unp.get(c1, c1)}_{chain_to_unp.get(c2, c2)}": v
            for (c1, c2), v in pair_dict.items()
        }

    cf_to_unp, ref_to_unp = invert_chain_mapping_dicts(row["chain_mapping"])

    cf_interfaces = {}
    ref_interfaces = {}

    if row["combfold_output_path"] is None:
        logger.warning(f"{row['complex_ac'] or '?'}: no combfold_output_path (n_combfold_outputs == 0) -- skipping.")
    else:
        assert Path(row["combfold_output_path"]).exists(), f"Combfold output path does not exist: {row['combfold_output_path']}"
        cf_interfaces = get_close_by_chain_pairs(row["combfold_output_path"], cutoff=cutoff, min_contacts=min_contacts)

    if row["reference_pdb_path"] is None:
        logger.warning(f"{row['complex_ac'] or '?'}: no reference_pdb_path -- skipping.")
    else:
        assert Path(row["reference_pdb_path"]).exists(), f"Reference PDB path does not exist: {row['reference_pdb_path']}"
        ref_interfaces = get_close_by_chain_pairs(row["reference_pdb_path"], cutoff=cutoff, min_contacts=min_contacts)

    return df_row.with_columns(
        pl.Series("cf_close_by_chainpairs", [map_chain_pairs_to_unp(cf_interfaces, cf_to_unp)], dtype=pl.Object),
        pl.Series("ref_close_by_chainpairs", [map_chain_pairs_to_unp(ref_interfaces, ref_to_unp)], dtype=pl.Object),
    )

In [20]:
df1 = pl.concat([
    _add_close_by_chain_pairs_columns(df1[i])
    for i in tqdm(range(df1.height), desc="Adding close-by chain pairs columns", unit="row")
])

Adding close-by chain pairs columns: 100%|██████████| 17/17 [00:18<00:00,  1.10s/row]


was the pdb in stoic?


In [21]:
stoic_data = pl.read_csv("/cluster/project/beltrao/kdammer/master_thesis/data/Stoic/data_file_stoic.csv")
stoic_training = stoic_data.filter(pl.col("split") == "train")

# is exactly this pdb in stoic
df1 = df1.with_columns(
    exact_pdb_in_stoic=pl.col("pdb_id").str.to_lowercase().is_in(
        stoic_training["pdb_id"].str.to_lowercase().implode()
    )
)


#is any pdb with these proteins in stoic training 
pdb_uniprot_mapping = pl.read_csv(
    "/cluster/project/beltrao/kdammer/master_thesis/data/pdb/pdb_chain_uniprot.csv",
    skip_rows=1,
    schema_overrides={
        "RES_BEG": pl.Utf8,
        "RES_END": pl.Utf8,
        "PDB_BEG": pl.Utf8,
        "PDB_END": pl.Utf8,
        "SP_BEG": pl.Utf8,
        "SP_END": pl.Utf8,
    },
)

#switch pdb ids to lowercase 
pdb_uniprot_mapping = pdb_uniprot_mapping.rename(
    {c: c.lower() for c in pdb_uniprot_mapping.columns}
)

#which prot ods are in each pdb
pdb_to_proteins = (
    pdb_uniprot_mapping
    .group_by("pdb")
    .agg(
        pl.col("sp_primary").unique().alias("proteins"),
        pl.col("chain").n_unique().alias("n_chains"),
    )
)


stoic_training = stoic_training.select(["pdb_id","split"])


#adds proteins and n_proteins to each pdb in stoic training (as stoic training only contains pdb ids)
stoic_training = stoic_training.join(
    pdb_to_proteins,
    left_on="pdb_id",
    right_on="pdb",
    how="left",
)


#get all proteins that are in that CP complex
df1 = df1.with_columns(
    pl.col("CP_stochiometry")
    .map_elements(lambda s: list(json.loads(s).keys()), return_dtype=pl.List(pl.Utf8))
    .alias("CP_stochiometry_protlist")
)

# canonicalize: sort list elements, join into a single string key
def canon(x):
    return x.list.sort().list.join(",")

in_stoic_training = set(canon(stoic_training["proteins"]).to_list())

df1 = df1.with_columns(
    canon(pl.col("CP_stochiometry_protlist"))
).with_columns(
    pl.col("CP_stochiometry_protlist").is_in(in_stoic_training).alias("pdb_with_identical_proteins_set_to_CF_in_stoic")
).drop("CP_stochiometry_protlist")


In [31]:
output_dir = Path(STEP_X3_OUTPUT_PATH).parent # This is again the "cf_pdb_structure_similarity" dir
assert output_dir.is_dir(), f"Output dir {output_dir} is not a directory"
obj_cols = [c for c, dt in zip(df1.columns, df1.dtypes) if dt == pl.Object]
df1 = df1.with_columns(
    pl.col(c).map_elements(lambda x: json.dumps(x) if x is not None else None, return_dtype=pl.Utf8)
    for c in obj_cols
)
df1.write_parquet(output_dir / "cf_pdb_eval_all_metrics.parquet")

In [22]:
from Bio.PDB import PDBParser, MMCIFParser, Superimposer
from Bio.Align import PairwiseAligner, substitution_matrices
from Bio.PDB.Polypeptide import is_aa
from Bio.SeqUtils import seq1
import numpy as np

def _get_chain_ca(struct_path: str, chain_id: str):
    """Returns (sequence_str, list[CA Atom]) for one chain, standard amino acids only."""
    parser = MMCIFParser(QUIET=True) if struct_path.endswith(".cif") else PDBParser(QUIET=True)
    structure = parser.get_structure("x", struct_path)
    chain = structure[0][chain_id]

    residues = [r for r in chain if is_aa(r, standard=True) and "CA" in r]
    seq = "".join(seq1(r.get_resname()) for r in residues)
    ca_atoms = [r["CA"] for r in residues]
    return seq, ca_atoms


def _aligned_ca_pairs(seq_cf, ca_cf, seq_pdb, ca_pdb):
    aligner = PairwiseAligner()
    aligner.mode = "global"
    aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")  
    aligner.open_gap_score = -10                                          
    aligner.extend_gap_score = -0.5                                       
    alignment = aligner.align(seq_cf, seq_pdb)[0]
    aligned_cf, aligned_pdb = alignment.indices

    matched_cf, matched_pdb, n_identical = [], [], 0
    for i_cf, i_pdb in zip(aligned_cf, aligned_pdb):
        if i_cf == -1 or i_pdb == -1:
            continue
        matched_cf.append(ca_cf[i_cf])
        matched_pdb.append(ca_pdb[i_pdb])
        n_identical += seq_cf[i_cf] == seq_pdb[i_pdb]

    identity = n_identical / len(matched_cf) if matched_cf else 0.0   # also guards the divide-by-zero
    return matched_cf, matched_pdb, identity

def compute_rmsd(combfold_output_path: str, reference_pdb_path: str, chain_mapping: dict[str, tuple[str, str]]):
    """chain_mapping: {uniprot: (cf_chain_id, pdb_chain_id)}. Returns (rmsd, min_identity)."""
    all_cf_atoms, all_pdb_atoms, identities = [], [], []

    for uniprot, (cf_chain, pdb_chain) in chain_mapping.items():
        seq_cf, ca_cf = _get_chain_ca(combfold_output_path, cf_chain)
        seq_pdb, ca_pdb = _get_chain_ca(reference_pdb_path, pdb_chain)
        assert seq_cf and seq_pdb, f"{uniprot}: empty sequence (cf={len(seq_cf)}, pdb={len(seq_pdb)})"

        matched_cf, matched_pdb, identity = _aligned_ca_pairs(seq_cf, ca_cf, seq_pdb, ca_pdb)
        assert matched_cf, f"{uniprot}: no aligned residues between CF chain {cf_chain} and PDB chain {pdb_chain}"

        all_cf_atoms += matched_cf
        all_pdb_atoms += matched_pdb
        identities.append(identity)

    sup = Superimposer()
    sup.set_atoms(all_pdb_atoms, all_cf_atoms)  # fixed=pdb, moving=cf
    return sup.rms, min(identities)